# SmolVLM Spline + MMR, MLP Top-32 and MLP + MMR at block 3

Select **GPU T4 ×2**, enable Internet, and use **Save & Run All**. The notebook downloads the verified after2 `checkpoints1/predictor_best.pt` from [attention-mlp, version 1](https://www.kaggle.com/models/z332k5/attention-mlp/other/default/1) through the public API, without attaching a Kaggle Input. To use another checkpoint source, clear `PREDICTOR_KAGGLE_MODEL` and set one of the alternative sources in the first code cell.

All three variants keep 32 of 384 visual tokens immediately after decoder block 3 (`layers[2]`). They run on the same 4,204 stratified DriveLM test questions with the same six-camera features, backbone, batch and FP32 greedy decoding. The two MMR variants use λ = 0.9, min–max relevance normalization and cosine redundancy from block-3 outputs.

- **Spline + MMR** subtracts a separate smoothing spline per camera from the attention at block 3. The fitted difference is block 3 minus block 23. Smoothing is selected on two disjoint recording groups from the 800 validation questions, then fit on all 800.
- **MLP Top-32** takes the 32 highest log probabilities from the frozen after2 MAP predictor, with ties decided by the original visual index.
- **MLP + MMR** uses the frozen after2 MAP predictor from the original branch, with the last prompt query and the visual hidden states after block 3. It uses the existing trained weights and interprets its log probabilities as relevance probabilities before MMR.

The report includes paired answers, selected token indices, quality and latency under the same timing procedure, and the spline plots. The comparison CSV includes score and answer changes relative to MLP Top-32.

In [ ]:
GPU_COUNT = 2
CALIBRATION_SIZE = 800
CALIBRATION_BATCH = 4  # Per GPU.
GENERATION_BATCH = 4   # Per GPU, with automatic CUDA OOM backoff.
KEEP_COUNTS = (32,)
CAMERA_BIAS_LAYERS = (3, 23)  # One-based decoder block numbers.
MMR_LAMBDA = 0.9
EVALUATION_SUBSET_SIZE = 4204
SELECTION_EXAMPLES = 3  # Save full ranked selections for the first evaluation examples.
CACHE_INPUT_DIR = ''   # Leave blank to fetch images and build features automatically.
PREDICTOR_KAGGLE_MODEL = 'z332k5/attention-mlp/other/default/1'
PREDICTOR_KAGGLE_MODEL_MEMBER = 'checkpoints1/predictor_best.pt'
PREDICTOR_KAGGLE_MODEL_SHA256 = '80b3be94ed4659c08123df290c49c330e77e1ff657f962153f62cdbfcfbc324f'
PREDICTOR_CHECKPOINT = ''  # Optional local replacement for the pinned after2 checkpoint.
PREDICTOR_HF_REPO = ''  # Alternative Hugging Face location for the existing predictor.
PREDICTOR_HF_REVISION = ''  # Required when using the Hugging Face option.
PREDICTOR_HF_FILENAME = 'predictor_best.pt'
PREDICTOR_HF_REPO_TYPE = 'model'
PREDICTOR_KAGGLE_RUN = ''  # e.g. 'owner/training-notebook-slug/3' for a saved version.
PREDICTOR_KAGGLE_FILE = 'predictor_best.pt'  # Can also name the training output ZIP.
PREDICTOR_KAGGLE_SECRET = ''  # Optional Kaggle Secret name holding an API token for private output.

In [ ]:
# Embedded source snapshot. Run this cell before importing Transformers.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from pathlib import Path
from base64 import b64decode
from io import BytesIO
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which('java') is None:
    raise RuntimeError('Java is required by the existing COCO caption evaluation metrics.')
SOURCE = Path(tempfile.mkdtemp(prefix='log-metrics-posprune-'))
SOURCE_B64 = '...'
with zipfile.ZipFile(BytesIO(b64decode(SOURCE_B64))) as archive:
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt'), 'pycocoevalcap==1.2'], check=True)
sys.path.insert(0, str(SOURCE))

In [ ]:
import torch
import transformers
import tokenizers
from huggingface_hub import hf_hub_download
from smolvlm_attention.config import MODELS, CACHE_DATASET, CACHE_FINGERPRINT, PREPARED_REPO, PREPARED_REVISION
from smolvlm_attention.posprune_ablation.cache_input import resolve_cache_input

if (transformers.__version__, tokenizers.__version__) != ('4.46.3', '0.20.4'):
    raise RuntimeError('Restart the kernel after dependency installation and Run All.')
visible_gpus = torch.cuda.device_count()
if visible_gpus == 0:
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
if GPU_COUNT < 1:
    raise ValueError('GPU_COUNT must be at least 1.')
worker_count = min(GPU_COUNT, visible_gpus)
print(f'Using {worker_count} of {visible_gpus} visible GPU(s).')

work = Path('/kaggle/working/smolvlm-posprune-spline-vs-mlp-after3-32')

from smolvlm_attention.posprune_ablation.predictor_input import resolve_predictor_input

predictor_checkpoint = resolve_predictor_input(
    work / 'predictor_input', checkpoint=PREDICTOR_CHECKPOINT, hf_repo=PREDICTOR_HF_REPO,
    hf_revision=PREDICTOR_HF_REVISION, hf_filename=PREDICTOR_HF_FILENAME, hf_repo_type=PREDICTOR_HF_REPO_TYPE,
    kaggle_run=PREDICTOR_KAGGLE_RUN, kaggle_file=PREDICTOR_KAGGLE_FILE, kaggle_secret=PREDICTOR_KAGGLE_SECRET,
    kaggle_model=PREDICTOR_KAGGLE_MODEL, kaggle_model_member=PREDICTOR_KAGGLE_MODEL_MEMBER,
    kaggle_model_sha256=PREDICTOR_KAGGLE_MODEL_SHA256)

cache_dir = None
if CACHE_INPUT_DIR:
    cache_dir = resolve_cache_input('/kaggle/input', CACHE_INPUT_DIR, dataset=CACHE_DATASET,
                                   fingerprint=CACHE_FINGERPRINT)
    print(f'Using mounted feature cache: {cache_dir}')
prepared_dir = work / 'prepared_data'
for name in ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl'):
    hf_hub_download(repo_id=PREPARED_REPO, repo_type='dataset', revision=PREPARED_REVISION,
                    filename=name, local_dir=prepared_dir, token=False)
base = MODELS['base']
checkpoint = hf_hub_download(repo_id=base['repo'], revision=base['revision'],
                             filename='finetune_best.pt', token=False)

In [ ]:
import json
cohort = json.loads((SOURCE / 'camera_posprune_cohort.json').read_bytes())

from smolvlm_attention.posprune_ablation.parallel import run_parallel

run_dir = Path(tempfile.mkdtemp(prefix='run-', dir=work))
output = run_parallel(checkpoint, cache_dir, prepared_dir, run_dir / 'results',
                      cache_fingerprint=CACHE_FINGERPRINT, camera_order=base['camera_order'], gpu_count=worker_count,
                      calibration_size=CALIBRATION_SIZE, evaluation_size=None, evaluation_fraction=cohort['evaluation_fraction'],
                      calibration_batch=CALIBRATION_BATCH, generation_batch=GENERATION_BATCH,
                      keep_counts=KEEP_COUNTS, seed=cohort['seed'],
                      selection_examples=SELECTION_EXAMPLES, camera_bias_layers=CAMERA_BIAS_LAYERS,
                      mmr_weight=MMR_LAMBDA, attention_normalization='minmax',
                      evaluation_subset_size=EVALUATION_SUBSET_SIZE, expected_evaluation_ids=cohort['evaluation_ids'],
                      predictor_checkpoint=predictor_checkpoint)
print((output / 'parallel_timing.json').read_text())
print(output)

In [ ]:
from IPython.display import FileLink, Image, display
from smolvlm_attention.posprune_ablation.report import report

for path in report(output):
    if path.suffix == '.png':
        display(Image(filename=str(path)))
archive = run_dir / 'posprune-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
tables = [output / name for name in ('indexed_selections.csv', 'comparison.csv', 'paired_answers.csv',
                                      'question_type_distribution.csv')]
for path in (*tables, archive):
    display(FileLink(os.path.relpath(path)))

display(FileLink(os.path.relpath(output / 'camera_fit_metrics.csv')))